In [1]:
import joblib
import sqlite3

In [2]:
model=joblib.load(r'C:\Users\Abhin\OneDrive\Documents\official project\ampicillin_model_standalone.joblib')

In [3]:
if hasattr(model, 'feature_names_in_'):
    features = list(model.feature_names_in_)
    print(f"✅ SUCCESS! Found {len(features)} exact feature names.")

✅ SUCCESS! Found 91 exact feature names.


In [4]:
print(features)

['blacmy', 'uhpt_e350q', 'estx', 'tet(c)', 'qnrb19', 'dfra17', 'qnrs1', 'dfra14', 'blatem', 'tet(d)', 'blactx-m-27', 'pare_s458a', 'ampc_t-32a', 'dfra12', 'pmrb_v161g', 'bladha-1', 'blatem-1', 'blacmy-2', 'tet(a)', 'parc_e84g', 'dfra1', 'aada1', 'erm(b)', 'cata1', 'blactx-m', 'dfra7', 'sat2', 'parc_s80i', 'blactx-m-55', 'aph(6)-id', 'catb3', 'ble', 'aac(3)-via', 'pare_d475e', 'pare_i529l', 'ampc_c-42t', 'blakpc-3', 'fosa3', 'marr_s3n', "aac(6')-ib-cr5", 'fosa7.5', 'parc_e84v', 'blacmy-4', 'blactx-m-32', 'blakpc-2', 'aac(3)-iid', 'emrd', 'blashv-1', 'tet(b)', 'gyra_d87y', 'blatem-19', "aph(3')-ia", 'pare_l416f', 'sul3', 'flor', 'estx-3', 'blaoxa-1', 'blatem-30', 'dfra8', 'gyra_d87n', 'ptsi_v25i', 'blatem-12', "aac(2')-iia", 'blaec', 'cyaa_s352t', 'qnrb4', 'rpob_h526y', 'blactx-m-15', 'nfsa_q67ter', 'blactx-m-14', 'mph(a)', 'blaher-3', 'aac(3)-iie', 'blaec-5', 'dfra5', 'gyra_s83l', 'cmla1', 'nfsa_q113ter', 'aada5', "aph(3'')-ib", 'mdtm', 'blatemp_c32t', 'dfra51', 'blactx-m-1', 'sul1', 'a

In [5]:
gene={}

In [6]:
for i in features:
    if i not in gene:
        j=input(f'Enter the dna sequence of current feature {i}')
        gene[i]=j


In [10]:
gene

{'blacmy': '',
 'uhpt_e350q': '',
 'estx': 'TGGATTGGCGGTCAAGCG',
 'tet(c)': 'ATGCGAATAGAAGCG',
 'qnrb19': 'ATGAAATCTAACAATGCGCTCATCGTCATCCTCGGCACCGTCACCCTGGATGCTGTAGGCATAGGCTTGGTTATGCCGGTACTGCCGGGC CTCTTGCGGGATATCGTCCATTCCGACAGCATCGCCAGTCACTATGGCGTGCTGCTAGCGCTATATGCGTTGATGCAATTTCTATGCGCA CCCGTTCTCGGAGCACTGTCCGACCGCTTTGGCCGCCGCCCAGTCCTGCTCGCTTCGCTACTTGGAGCCACTATCGACTACGCGATCATG GCGACCACACCCGTCCTGTGGATCCTCTACGCCGGACGCATCGTGGCCGGCATCACCGGCGCCACAGGTGCGGTTGCTGGCGCCTATATC GCCGACATCACCGATGGGGAAGATCGGGCTCGCCACTTCGGGCTCATGAGCGCTTGTTTCGGCGTGGGTATGGTGGCAGGCCCCGTGGCC GGGGGACTGTTGGGCGCCATCTCCTTGCATGCACCATTCCTTGCGGCGGCGGTGCTCAACGGCCTCAACCTACTACTGGGCTGCTTCCTA ATGCAGGAGTCGCATAAGGGAGAGCGTCGACCGATGCCCTTGAGAGCCTTCAACCCAGTCAGCTCCTTCCGGTGGGCGCGGGGCATGACT ATCGTCGCCGCACTTATGACTGTCTTCTTTATCATGCAACTCGTAGGACAGGTGCCGGCAGCGCTCTGGGTCATTTTCGGCGAGGACCGC TTTCGCTGGAGCGCGACGATGATCGGCCTGTCGCTTGCGGTATTCGGAATCTTGCACGCCCTCGCTCAAGCCTTCGTCACTGGTCCCGCC ACCAAACGTTTCGGCGAGAAGCAGGCCATTATCGCCGGCATGGCGGCCGACGCGCTGGGCTACGTCTTGCTGGC

In [8]:
gene['blatem']='TTGAAAGTATCATTGATGGCTGCGAAAGCGAAAAACGGCGTGATTGGTTGCGGTCCAGACATACCCTGGTCCGCGAAAGGGGAGCAGCTACTTTTTAAAGCATTGACCTACAATCAGTGGCTTCTGGTGGGTCGCAAGACGTTTGAATCTATGGGCGCACTCCCCAATAGGAAATACGCGGTCGTTACCCGCTCAGGTTGGACATCAAATGATGACAATGTAGTTGTATTTCAGTCAATCGAAGAGGCCATGGACAGGCTAGCTGAATTCACCGGTCACGTTATAGTGTCTGGTGGCGGAGAAATTTACCGAGAAACATTACCCATGGCCTCTACGCTCCACTTATCGACGATCGACATCGAGCCAGAGGGGGATGTTTTCTTCCCGAGTATTCCAAATACCTTCGAAGTTGTTTTTGAGCAACACTTTACTTCAAACATTAACTATTGCTATCAAATTTGGAAAAAGGGTTAA'

In [9]:
import sqlite3

# 1. Your completed dictionary


# 2. Connect to the database (this creates the file if it doesn't exist)
conn = sqlite3.connect('resistance_genes.db')
cursor = conn.cursor()

# 3. Create the table
cursor.execute('''
CREATE TABLE IF NOT EXISTS sequences (
    gene_name TEXT PRIMARY KEY,
    dna_sequence TEXT
)
''')

# 4. Insert the dictionary data into the database
for name, seq in gene.items():
    cursor.execute('''
    INSERT OR REPLACE INTO sequences (gene_name, dna_sequence)
    VALUES (?, ?)
    ''', (name, seq))

# 5. Save and close
conn.commit()
conn.close()

print("Database created successfully!")

Database created successfully!


In [11]:
import sqlite3
from Bio import Align

def check_sequence_in_db(target_gene, query_seq):
    # 1. Connect to the database
    conn = sqlite3.connect('resistance_genes.db')
    cursor = conn.cursor()
    
    # 2. Fetch the stored sequence for the specific gene
    cursor.execute("SELECT dna_sequence FROM sequences WHERE gene_name = ?", (target_gene,))
    result = cursor.fetchone()
    conn.close()
    
    if not result:
        print(f"Gene '{target_gene}' not found in database.")
        return

    db_seq = result[0].upper().strip()
    query_seq = query_seq.upper().strip()
    
    print(f"--- Comparison for {target_gene} ---")
    
    # 3. Exact Match Check
    if query_seq in db_seq or db_seq in query_seq:
        print("RESULT: Exact Match Found!")
    else:
        # 4. Alignment Fallback (Matches ARIS logic)
        aligner = Align.PairwiseAligner()
        aligner.mode = 'local'
        aligner.match_score = 1
        aligner.mismatch_score = -1
        
        score = aligner.score(query_seq, db_seq)
        max_score = len(db_seq)
        match_perc = (score / max_score) * 100
        
        print(f"RESULT: No exact match. Alignment Score: {match_perc:.2f}%")
        if match_perc >= 95.0:
            print("STATUS: Sequence is valid (within 95% threshold).")
        else:
            print("STATUS: Sequence is too different from database record.")

# Example Usage
my_sequence = "ATGAGTATTCAACATTTTCGTGTCGCCCTTATTCC..." # Paste full seq here
check_sequence_in_db("blatem-1", my_sequence)

--- Comparison for blatem-1 ---
RESULT: No exact match. Alignment Score: 1.14%
STATUS: Sequence is too different from database record.


In [12]:
import sqlite3

def add_gene_to_db(gene_name, dna_sequence):
    try:
        # Connect to your existing database file
        conn = sqlite3.connect('resistance_genes.db')
        cursor = conn.cursor()

        # SQL command to insert a new row
        # Using '?' as placeholders prevents SQL injection
        sql = "INSERT INTO sequences (gene_name, dna_sequence) VALUES (?, ?)"
        
        # Clean and format the data before inserting
        data = (gene_name.strip(), dna_sequence.strip().upper())

        cursor.execute(sql, data)
        
        # You MUST commit to save the changes to the file
        conn.commit()
        print(f"Successfully added {gene_name} to the database.")

    except sqlite3.IntegrityError:
        print(f"Error: The gene '{gene_name}' already exists in the database.")
    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        conn.close()

# Example: Adding a new marker
new_gene = "blaTEM-1_variant"
new_seq = "ATGAGTATTCAACATTTTCGTGTCGCCCTTATTCC..." # Paste your DNA here
add_gene_to_db(new_gene, new_seq)

Successfully added blaTEM-1_variant to the database.


In [13]:
import sqlite3

def get_blatem_sequence():
    try:
        # Connect to your database
        conn = sqlite3.connect('resistance_genes.db')
        cursor = conn.cursor()
        
        # Query specifically for the 'blatem' sequence
        query = "SELECT dna_sequence FROM sequences WHERE gene_name = 'blatem'"
        cursor.execute(query)
        
        result = cursor.fetchone()
        conn.close()
        
        if result:
            print("--- Sequence for 'blatem' ---")
            print(result[0])
            print(f"\nTotal Length: {len(result[0])} bp")
        else:
            print("Error: Gene 'blatem' not found in database.")
            
    except Exception as e:
        print(f"An error occurred: {e}")

if __name__ == "__main__":
    get_blatem_sequence()

--- Sequence for 'blatem' ---
TTGAAAGTATCATTGATGGCTGCGAAAGCGAAAAACGGCGTGATTGGTTGCGGTCCAGACATACCCTGGTCCGCGAAAGGGGAGCAGCTACTTTTTAAAGCATTGACCTACAATCAGTGGCTTCTGGTGGGTCGCAAGACGTTTGAATCTATGGGCGCACTCCCCAATAGGAAATACGCGGTCGTTACCCGCTCAGGTTGGACATCAAATGATGACAATGTAGTTGTATTTCAGTCAATCGAAGAGGCCATGGACAGGCTAGCTGAATTCACCGGTCACGTTATAGTGTCTGGTGGCGGAGAAATTTACCGAGAAACATTACCCATGGCCTCTACGCTCCACTTATCGACGATCGACATCGAGCCAGAGGGGGATGTTTTCTTCCCGAGTATTCCAAATACCTTCGAAGTTGTTTTTGAGCAACACTTTACTTCAAACATTAACTATTGCTATCAAATTTGGAAAAAGGGTTAA

Total Length: 474 bp


In [14]:
from Bio import SeqIO
import io

# Paste your file content here to test
data = """>Test_Sample
TTGAAAGTATCATTGATGGCTGCGAAAGCGAAAAACGGCGTGATTGGTTGCGGTCCAGACATACCCTGGTCCGCGAAAGGGGAGCAGCTACTTTTTAAAGCATTGACCTACAATCAGTGGCTTCTGGTGGGTCGCAAGACGTTTGAATCTATGGGCGCACTCCCCAATAGGAAATACGCGGTCGTTACCCGCTCAGGTTGGACATCAAATGATGACAATGTAGTTGTATTTCAGTCAATCGAAGAGGCCATGGACAGGCTAGCTGAATTCACCGGTCACGTTATAGTGTCTGGTGGCGGAGAAATTTACCGAGAAACATTACCCATGGCCTCTACGCTCCACTTATCGACGATCGACATCGAGCCAGAGGGGGATGTTTTCTTCCCGAGTATTCCAAATACCTTCGAAGTTGTTTTTGAGCAACACTTTACTTCAAACATTAACTATTGCTATCAAATTTGGAAAAAGGGTTAA"""

stream = io.StringIO(data)
records = list(SeqIO.parse(stream, "fasta"))

if not records:
    print("❌ FAILED: No records found. Check your FASTA header (the > line).")
else:
    for record in records:
        print(f"✅ SUCCESS: Found record ID: {record.id}")
        print(f"Sequence Length: {len(record.seq)} bp")

✅ SUCCESS: Found record ID: Test_Sample
Sequence Length: 474 bp
